# KHAI PHÁ DỮ LIỆU REVIEW TIKI 9K ĐÃ GÁN NHÃN

Notebook được xây dựng theo cấu trúc của `Khai_pha_du_lieu.ipynb`, tập trung:

- Đọc và kiểm tra dữ liệu review đã gán nhãn.
- Chỉ sử dụng những trường thực tế có trong raw data.
- Tạo feature độ dài review và đặc trưng thời gian.
- Phân tích ảnh hưởng của rating, độ dài nội dung, helpful vote và thời gian đến nhãn `help`.

## Bước 1: Chuẩn bị môi trường và dữ liệu

In [1]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

%matplotlib inline
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
pd.set_option("display.max_colwidth", 160)

DATA_PATH = Path("Data/gold_data/tiki_reviews_full_labeled_9k.csv")
if not DATA_PATH.exists():
    matches = list(Path(".").rglob("tiki_reviews_full_labeled_9k.csv"))
    if not matches:
        raise FileNotFoundError("Không tìm thấy tiki_reviews_full_labeled_9k.csv")
    DATA_PATH = matches[0]

print("Dữ liệu:", DATA_PATH.resolve())

Dữ liệu: /Users/dphong2005/Desktop/Demo_DS/Data/gold_data/tiki_reviews_full_labeled_9k.csv


In [2]:
df = pd.read_csv(DATA_PATH)

available_columns = [
    "review_id", "product_id", "user_id", "rating", "title", "content",
    "created_at", "helpful_count", "purchased", "crawled_at", "help"
]
missing = [col for col in available_columns if col not in df.columns]
if missing:
    raise ValueError(f"Thiếu cột cần thiết: {missing}")

df = df[available_columns].copy()
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["content"] = df["content"].fillna("").astype(str).str.strip()
df["review_context"] = (
    df["title"] + ". " + df["content"]
).str.strip(". ")

df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["crawled_at"] = pd.to_datetime(df["crawled_at"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["helpful_count"] = pd.to_numeric(df["helpful_count"], errors="coerce").fillna(0)
df["help"] = pd.to_numeric(df["help"], errors="raise").astype(int)
df["purchased"] = df["purchased"].astype(bool)

# Feature engineering chỉ từ dữ liệu quan sát được.
df["review_char_length"] = df["review_context"].str.len()
df["review_word_length"] = df["review_context"].str.split().str.len()
df["title_char_length"] = df["title"].str.len()
df["content_char_length"] = df["content"].str.len()
df["sentence_count"] = df["review_context"].str.count(r"[.!?]+").clip(lower=1)
df["exclamation_count"] = df["review_context"].str.count("!")
df["question_count"] = df["review_context"].str.count(r"\?")
df["digit_count"] = df["review_context"].str.count(r"\d")
df["created_year"] = df["created_at"].dt.year
df["created_month"] = df["created_at"].dt.month
df["created_hour"] = df["created_at"].dt.hour
df["created_weekday"] = df["created_at"].dt.day_name()
df["has_helpful_vote"] = (df["helpful_count"] > 0).astype(int)

print(f"Kích thước sau khi tạo feature: {df.shape[0]:,} dòng x {df.shape[1]} cột")
display(df.head())

Kích thước sau khi tạo feature: 9,000 dòng x 25 cột


,review_id,product_id,user_id,rating,title,content,created_at,helpful_count,purchased,crawled_at,...,content_char_length,sentence_count,exclamation_count,question_count,digit_count,created_year,created_month,created_hour,created_weekday,has_helpful_vote
0,4839635,36626940,10852,5,Cực kì hài lòng,"Giao hàng nhanh chóng, sản phẩm đúng như quảng cáo. Đã sử dụng thử, khá tiện lợi!",2020-09-13 11:23:16,0,False,2026-05-05 22:48:50.033785,...,81,3,1,0,0,2020,9,11,Sunday,0
1,19814968,11183003,28493267,5,Cực kì hài lòng,Sp tốt. Rất hài lòng về cách phục vụ của shop,2024-03-21 09:18:54,0,False,2026-05-07 10:11:45.486510,...,45,2,0,0,0,2024,3,9,Thursday,0
2,12513167,59504295,6507390,5,Cực kì hài lòng,tot,2021-10-09 09:02:31,0,False,2026-05-06 17:14:07.192966,...,3,1,0,0,0,2021,10,9,Saturday,0
3,14825710,118137722,18115728,5,Cực kì hài lòng,Tuyệt vời ưng ý!,2022-02-07 08:16:39,0,False,2026-05-05 20:05:37.145608,...,16,2,1,0,0,2022,2,8,Monday,0
4,10951398,631034,18356094,5,Cực kì hài lòng,cực kỳ hài lòng sp,2021-07-07 08:33:48,0,False,2026-05-05 23:57:41.530019,...,18,1,0,0,0,2021,7,8,Wednesday,0


## Bước 2: Tổng quan cấu trúc dữ liệu

In [3]:
print("THÔNG TIN TỔNG QUAN")
print("=" * 70)
print(f"Số review: {len(df):,}")
print(f"Số cột sau feature engineering: {df.shape[1]}")
print(f"Dung lượng bộ nhớ: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Khoảng thời gian review: {df['created_at'].min()} -> {df['created_at'].max()}")
print(f"Số sản phẩm: {df['product_id'].nunique():,}")
print(f"Số người dùng: {df['user_id'].nunique():,}")
df.info()

THÔNG TIN TỔNG QUAN
Số review: 9,000
Số cột sau feature engineering: 25
Dung lượng bộ nhớ: 7.34 MB
Khoảng thời gian review: 2014-12-17 15:06:13 -> 2026-04-29 14:21:16
Số sản phẩm: 797
Số người dùng: 8,812
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   review_id            9000 non-null   int64         
 1   product_id           9000 non-null   int64         
 2   user_id              9000 non-null   int64         
 3   rating               9000 non-null   int64         
 4   title                9000 non-null   object        
 5   content              9000 non-null   object        
 6   created_at           9000 non-null   datetime64[ns]
 7   helpful_count        9000 non-null   int64         
 8   purchased            9000 non-null   bool          
 9   crawled_at           9000 non-null   datetime64[ns]
 10 

## Bước 3: Kiểm tra chất lượng dữ liệu

In [4]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_rate_pct": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=True),
})
display(quality)

print("Duplicate review_id:", int(df["review_id"].duplicated().sum()))
print("Duplicate review_context:", int(df["review_context"].duplicated().sum()))
print("Review rỗng:", int(df["review_context"].eq("").sum()))
print("Nhãn ngoài 0/1:", int((~df["help"].isin([0, 1])).sum()))

,dtype,missing_count,missing_rate_pct,unique_count
review_id,int64,0,0.0,9000
product_id,int64,0,0.0,797
user_id,int64,0,0.0,8812
rating,int64,0,0.0,5
title,object,0,0.0,701
content,object,0,0.0,7344
created_at,datetime64[ns],0,0.0,8999
helpful_count,int64,0,0.0,42
purchased,bool,0,0.0,1
crawled_at,datetime64[ns],0,0.0,797


Duplicate review_id: 0
Duplicate review_context: 1616
Review rỗng: 0
Nhãn ngoài 0/1: 0


## Bước 4: Thống kê mô tả các biến số

In [5]:
numeric_columns = [
    "rating", "helpful_count", "help", "review_char_length",
    "review_word_length", "title_char_length", "content_char_length",
    "sentence_count", "exclamation_count", "question_count", "digit_count",
]
display(df[numeric_columns].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).T)

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
rating,9000.0,4.202111,1.382400,1.0,4.0,5.0,5.0,5.0,5.00,5.00,5.0
helpful_count,9000.0,0.558556,2.939648,0.0,0.0,0.0,0.0,1.0,3.00,10.00,104.0
help,9000.0,0.559778,0.496441,0.0,0.0,1.0,1.0,1.0,1.00,1.00,1.0
review_char_length,9000.0,99.735667,119.816920,8.0,29.0,72.0,125.0,198.0,277.05,549.01,4184.0
review_word_length,9000.0,22.641333,26.934022,2.0,7.0,16.0,28.0,45.0,63.00,125.00,924.0
title_char_length,9000.0,14.703778,5.228614,1.0,15.0,15.0,15.0,18.0,18.00,34.00,128.0
content_char_length,9000.0,83.310667,119.184622,0.0,13.0,56.0,109.0,180.0,259.05,528.03,4167.0
sentence_count,9000.0,1.814556,1.485175,1.0,1.0,1.0,2.0,4.0,4.00,7.00,41.0
exclamation_count,9000.0,0.094111,0.541149,0.0,0.0,0.0,0.0,0.0,1.00,2.00,21.0
question_count,9000.0,0.082333,0.991014,0.0,0.0,0.0,0.0,0.0,0.00,2.00,44.0


## Bước 5: Phân phối nhãn và rating

In [6]:
label_summary = (
    df["help"].value_counts().sort_index()
    .rename_axis("help").reset_index(name="count")
)
label_summary["percentage"] = (label_summary["count"] / len(df) * 100).round(2)
display(label_summary)

rating_summary = (
    df.groupby("rating")
    .agg(
        review_count=("review_id", "size"),
        helpful_count=("help", "sum"),
        helpful_rate=("help", "mean"),
        avg_word_length=("review_word_length", "mean"),
    )
    .reset_index()
)
rating_summary["helpful_rate_pct"] = (rating_summary["helpful_rate"] * 100).round(2)
display(rating_summary)

,help,count,percentage
0,0,3962,44.02
1,1,5038,55.98


,rating,review_count,helpful_count,helpful_rate,avg_word_length,helpful_rate_pct
0,1,1034,832,0.804642,29.586074,80.46
1,2,442,352,0.796380,29.762443,79.64
2,3,343,277,0.807580,30.827988,80.76
3,4,1033,602,0.582769,22.642788,58.28
4,5,6148,2975,0.483897,20.504392,48.39


## Bước 6: Phân tích độ dài review theo nhãn

In [7]:
length_summary = (
    df.groupby("help")
    .agg(
        review_count=("review_id", "size"),
        avg_char_length=("review_char_length", "mean"),
        median_char_length=("review_char_length", "median"),
        avg_word_length=("review_word_length", "mean"),
        median_word_length=("review_word_length", "median"),
        avg_sentence_count=("sentence_count", "mean"),
    )
    .round(2)
)
display(length_summary)

length_bins = pd.cut(
    df["review_word_length"],
    bins=[-1, 5, 10, 20, 40, 80, np.inf],
    labels=["1-5", "6-10", "11-20", "21-40", "41-80", ">80"],
)
length_effect = (
    df.assign(length_group=length_bins)
    .groupby("length_group", observed=False)
    .agg(
        review_count=("review_id", "size"),
        helpful_count=("help", "sum"),
        helpful_rate=("help", "mean"),
    )
    .reset_index()
)
length_effect["helpful_rate_pct"] = (length_effect["helpful_rate"] * 100).round(2)
display(length_effect)

,review_count,avg_char_length,median_char_length,avg_word_length,median_word_length,avg_sentence_count
help,,,,,,
0,3962,42.92,25.0,9.85,6.0,1.25
1,5038,144.41,112.0,32.70,25.0,2.26


,length_group,review_count,helpful_count,helpful_rate,helpful_rate_pct
0,1-5,1824,38,0.020833,2.08
1,6-10,1424,358,0.251404,25.14
2,11-20,1931,1237,0.640601,64.06
3,21-40,2696,2329,0.863872,86.39
4,41-80,871,826,0.948335,94.83
5,>80,254,250,0.984252,98.43


## Bước 7: Helpful vote, purchased và nhãn help

In [8]:
vote_summary = (
    df.groupby("has_helpful_vote")
    .agg(
        review_count=("review_id", "size"),
        avg_platform_helpful_count=("helpful_count", "mean"),
        labeled_helpful_count=("help", "sum"),
        labeled_helpful_rate=("help", "mean"),
    )
    .round(3)
)
display(vote_summary)

purchased_summary = (
    df.groupby("purchased")
    .agg(
        review_count=("review_id", "size"),
        helpful_rate=("help", "mean"),
        avg_word_length=("review_word_length", "mean"),
    )
    .round(3)
)
display(purchased_summary)

,review_count,avg_platform_helpful_count,labeled_helpful_count,labeled_helpful_rate
has_helpful_vote,,,,
0,7600,0.000,3930,0.517
1,1400,3.591,1108,0.791


,review_count,helpful_rate,avg_word_length
purchased,,,
False,9000,0.56,22.641


## Bước 8: Phân tích thời gian tạo review

In [9]:
year_summary = (
    df.dropna(subset=["created_year"])
    .groupby("created_year")
    .agg(
        review_count=("review_id", "size"),
        helpful_rate=("help", "mean"),
        avg_word_length=("review_word_length", "mean"),
    )
    .reset_index()
)
year_summary["helpful_rate_pct"] = (year_summary["helpful_rate"] * 100).round(2)
display(year_summary)

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]
weekday_summary = (
    df.groupby("created_weekday")
    .agg(review_count=("review_id", "size"), helpful_rate=("help", "mean"))
    .reindex(weekday_order)
)
weekday_summary["helpful_rate_pct"] = (weekday_summary["helpful_rate"] * 100).round(2)
display(weekday_summary)

,created_year,review_count,helpful_rate,avg_word_length,helpful_rate_pct
0,2014,1,1.000000,69.000000,100.00
1,2015,5,1.000000,105.000000,100.00
2,2016,13,1.000000,106.923077,100.00
3,2017,81,0.864198,48.913580,86.42
4,2018,192,0.875000,42.213542,87.50
5,2019,573,0.825480,38.762653,82.55
6,2020,1428,0.662465,28.007703,66.25
7,2021,3366,0.515746,19.413844,51.57
8,2022,2453,0.448430,16.784346,44.84
9,2023,530,0.583019,23.388679,58.30


,review_count,helpful_rate,helpful_rate_pct
created_weekday,,,
Monday,1402,0.569900,56.99
Tuesday,1317,0.567198,56.72
Wednesday,1349,0.550037,55.00
Thursday,1328,0.576054,57.61
Friday,1342,0.573025,57.30
Saturday,1137,0.543536,54.35
Sunday,1125,0.531556,53.16


## Bước 9: Tương quan giữa các biến số

In [10]:
correlation_columns = [
    "rating", "helpful_count", "purchased", "review_char_length",
    "review_word_length", "sentence_count", "exclamation_count",
    "question_count", "digit_count", "created_year", "created_month",
    "created_hour", "help",
]
correlation_matrix = df[correlation_columns].astype(float).corr()
display(correlation_matrix.round(3))

target_corr = (
    correlation_matrix["help"]
    .drop("help")
    .sort_values(key=lambda series: series.abs(), ascending=False)
    .rename("correlation_with_help")
)
display(target_corr.to_frame())

,rating,helpful_count,purchased,review_char_length,review_word_length,sentence_count,exclamation_count,question_count,digit_count,created_year,created_month,created_hour,help
rating,1.000,-0.060,NaN,-0.134,-0.131,-0.087,-0.024,-0.065,-0.163,0.014,-0.028,-0.004,-0.246
helpful_count,-0.060,1.000,NaN,0.284,0.284,0.227,0.125,0.017,0.130,-0.169,-0.013,0.004,0.112
purchased,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_char_length,-0.134,0.284,NaN,1.000,0.998,0.778,0.148,0.057,0.476,-0.205,-0.003,-0.020,0.421
review_word_length,-0.131,0.284,NaN,0.998,1.000,0.775,0.144,0.052,0.477,-0.203,-0.003,-0.019,0.421
sentence_count,-0.087,0.227,NaN,0.778,0.775,1.000,0.233,0.100,0.413,-0.171,0.007,-0.020,0.338
exclamation_count,-0.024,0.125,NaN,0.148,0.144,0.233,1.000,0.009,0.056,-0.064,0.002,-0.005,0.054
question_count,-0.065,0.017,NaN,0.057,0.052,0.100,0.009,1.000,0.034,-0.055,0.012,-0.003,0.023
digit_count,-0.163,0.130,NaN,0.476,0.477,0.413,0.056,0.034,1.000,-0.092,0.012,-0.031,0.251
created_year,0.014,-0.169,NaN,-0.205,-0.203,-0.171,-0.064,-0.055,-0.092,1.000,-0.210,-0.013,-0.161


,correlation_with_help
review_word_length,0.421293
review_char_length,0.420505
sentence_count,0.337645
digit_count,0.251485
rating,-0.246159
created_year,-0.160995
helpful_count,0.111552
exclamation_count,0.054132
question_count,0.023311
created_hour,-0.013635


## Bước 10: Quan sát các review đại diện

In [11]:
for label in [0, 1]:
    print(f"\n===== SAMPLE help={label} =====")
    sample = df[df["help"] == label].sample(5, random_state=42)
    display(sample[
        [
            "review_id", "rating", "review_context", "review_word_length",
            "helpful_count", "created_at", "help",
        ]
    ])


===== SAMPLE help=0 =====


,review_id,rating,review_context,review_word_length,helpful_count,created_at,help
277,11738127,5,Cực kì hài lòng. Good,5,0,2021-08-11 04:44:43,0
2166,19472508,5,Cực kì hài lòng. Tốt,5,0,2023-08-20 07:21:54,0
2793,13539396,5,Cực kì hài lòng. Ok,5,0,2021-11-30 13:32:09,0
1417,18674650,5,Cực kì hài lòng. Giao hàng nhanh\r\nGiá tốt,9,0,2023-01-23 04:25:41,0
594,9456406,5,"Cực kì hài lòng. sách đẹp, giấy đẹp, giao hàng nhanh, nhưng nội dung bình thường không hấp dẫn lắm",20,0,2021-05-02 07:28:48,0



===== SAMPLE help=1 =====


,review_id,rating,review_context,review_word_length,helpful_count,created_at,help
8567,18509464,5,"Cực kì hài lòng. Của Trung Nguyên thì thơm ngon rồi, tuy nhiên lần mua này đóng gói ko biết do khâu nào mà hộp bẹp, hở rộng.. may ko ảnh hưởng đến hàng bên ...",36,0,2022-12-26 08:54:11,1
5894,2440513,5,"sản phẩm đẹp, hài lòng. balo đẹp, chắc chắn, vải bố dày, có 2 ngăn lớn khá rộng, mỗi ngăn đều có miếng lót xốp chống sốc, để vừa laptop 15.6"", tiki giao hàn...",57,0,2019-09-10 15:06:44,1
5219,10801663,4,"Hài lòng. giao hàng siêu lâu, có túi tặng kèm như quảng cáo, hộp đóng gói móp méo",18,0,2021-07-01 13:51:31,1
5559,6083847,5,"Cực kì hài lòng. Sách đẹp, nội dung thấm. Đáng đọc với những bạn ở độ tuổi trưởng thành",19,0,2020-12-10 14:50:11,1
6024,13375589,5,Cực kì hài lòng. Máy hút chất lượng tốt trong tầm giá,12,0,2021-11-22 16:52:22,1


## Kết luận

- Nhãn `help` cần được phân tích cùng với **nội dung review**, không chỉ rating.
- Độ dài review là feature quan trọng để khảo sát, nhưng review dài không mặc định hữu ích.
- `helpful_count` của nền tảng là tín hiệu tham khảo, không phải nhãn thay thế.
- Các feature thời gian mô tả hành vi dữ liệu nhưng cần tránh diễn giải thành quan hệ nhân quả.